# Embed Infocom.am Investigation Chunks

Embeds article chunks using **Metric-AI/armenian-text-embeddings-2-large** (1024d) on a free T4 GPU.

**Steps:**
1. Run all cells (repo is cloned automatically)
2. Copy output files from `scraped_data/` to your local machine
3. Run `python load_embeddings.py --strategy paragraph` locally

**Runtime:** ~2 min for 421 paragraph chunks on T4 GPU

In [1]:
print(509)

509


In [ ]:
# If repo already cloned from paragraph run, just cd into it
# Otherwise uncomment the git clone line
!git clone https://github.com/HaykTarkhanyan/rag_workflow.git
%cd /content/rag_workflow
!git pull

In [2]:
!git clone https://github.com/HaykTarkhanyan/rag_workflow.git
%cd /content/rag_workflow
!git pull

Cloning into 'rag_workflow'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 101 (delta 42), reused 81 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 785.52 KiB | 3.03 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/rag_workflow
Already up to date.


In [3]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


In [4]:
# Load chunks from cloned repo
import json

STRATEGY = "sentence"  # Changed from "paragraph" -- already done

filename = f"scraped_data/chunks_{STRATEGY}.json"
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]
print(f"Loaded {len(chunks)} chunks ({STRATEGY} strategy)")
print(f"Sample chunk_id: {chunks[0]['chunk_id']}")

Loaded 1390 chunks (sentence strategy)
Sample chunk_id: infocom_10007787_chunk_000


In [5]:
# Load model
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import time
import numpy as np

MODEL_NAME = "Metric-AI/armenian-text-embeddings-2-large"
BATCH_SIZE = 32  # GPU can handle larger batches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")

Using device: cuda
Loading Metric-AI/armenian-text-embeddings-2-large...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded in 51.8s


In [6]:
# Embedding function
def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def embed_texts(texts, tokenizer, model, device, batch_size=BATCH_SIZE):
    all_embeddings = []
    t0 = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        batch_dict = tokenizer(
            batch, max_length=512, padding=True, truncation=True, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**batch_dict)

        embeddings = average_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())

        done = min(i + batch_size, len(texts))
        elapsed = time.time() - t0
        rate = done / elapsed
        eta = (len(texts) - done) / rate if rate > 0 else 0
        print(f"  [{done}/{len(texts)}] {elapsed:.1f}s elapsed, ~{eta:.0f}s remaining")

    return np.vstack(all_embeddings)

print("Embedding function ready")

Embedding function ready


In [7]:
# Embed all chunks
texts = [c["text_for_embedding"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

print(f"Embedding {len(texts)} chunks...")
t0 = time.time()
embeddings = embed_texts(texts, tokenizer, model, device)
total = time.time() - t0
print(f"\nDone! {len(texts)} chunks in {total:.1f}s ({total/len(texts)*1000:.0f}ms/chunk)")
print(f"Embedding shape: {embeddings.shape}")

Embedding 1390 chunks...
  [32/1390] 2.1s elapsed, ~87s remaining
  [64/1390] 3.1s elapsed, ~65s remaining
  [96/1390] 4.2s elapsed, ~57s remaining
  [128/1390] 5.3s elapsed, ~53s remaining
  [160/1390] 6.6s elapsed, ~50s remaining
  [192/1390] 8.0s elapsed, ~50s remaining
  [224/1390] 9.3s elapsed, ~49s remaining
  [256/1390] 10.5s elapsed, ~47s remaining
  [288/1390] 11.7s elapsed, ~45s remaining
  [320/1390] 13.0s elapsed, ~43s remaining
  [352/1390] 14.2s elapsed, ~42s remaining
  [384/1390] 15.4s elapsed, ~40s remaining
  [416/1390] 16.9s elapsed, ~40s remaining
  [448/1390] 17.9s elapsed, ~38s remaining
  [480/1390] 19.1s elapsed, ~36s remaining
  [512/1390] 21.6s elapsed, ~37s remaining
  [544/1390] 22.8s elapsed, ~35s remaining
  [576/1390] 24.0s elapsed, ~34s remaining
  [608/1390] 25.2s elapsed, ~32s remaining
  [640/1390] 26.3s elapsed, ~31s remaining
  [672/1390] 27.5s elapsed, ~29s remaining
  [704/1390] 28.6s elapsed, ~28s remaining
  [736/1390] 29.8s elapsed, ~26s remain

In [8]:
# Quick sanity check: similarity between first 2 chunks of same article
sim = np.dot(embeddings[0], embeddings[1])
print(f"Similarity between chunk 0 and 1 (same article): {sim:.4f}")

# Find most different chunk from chunk 0
sims = embeddings @ embeddings[0]
most_different = np.argmin(sims)
print(f"Most different from chunk 0: chunk {most_different} (sim={sims[most_different]:.4f})")
print(f"  -> {chunks[most_different]['metadata']['title'][:70]}")

Similarity between chunk 0 and 1 (same article): 0.8836
Most different from chunk 0: chunk 1207 (sim=0.3321)
  -> «Evolution»-ը բուհերում. ինչպես են ուսանողներին խրախուսում դառնալ ազար


In [9]:
# Save embeddings + metadata into scraped_data/
output_prefix = f"scraped_data/embeddings_{STRATEGY}"

# Save numpy array
np.save(f"{output_prefix}.npy", embeddings)
print(f"Saved embeddings to {output_prefix}.npy ({embeddings.nbytes / 1024**2:.1f} MB)")

# Save chunk_id -> index mapping and metadata
meta_output = {
    "strategy": STRATEGY,
    "model": MODEL_NAME,
    "embedding_dim": int(embeddings.shape[1]),
    "total_chunks": len(chunks),
    "chunk_ids": chunk_ids,
    "chunks": [
        {
            "chunk_id": c["chunk_id"],
            "article_id": c["article_id"],
            "chunk_index": c["chunk_index"],
            "total_chunks": c["total_chunks"],
            "text": c["text"],
            "metadata": c["metadata"],
        }
        for c in chunks
    ],
}

with open(f"{output_prefix}_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta_output, f, ensure_ascii=False, indent=2)
print(f"Saved metadata to {output_prefix}_meta.json")
print(f"\nCopy these files to your local scraped_data/ folder, then run:")
print(f"  python load_embeddings.py --strategy {STRATEGY}")

Saved embeddings to scraped_data/embeddings_sentence.npy (5.4 MB)
Saved metadata to scraped_data/embeddings_sentence_meta.json

Copy these files to your local scraped_data/ folder, then run:
  python load_embeddings.py --strategy sentence
